# ZOS — Feature Engineering & Importance

Builds a **70-feature customer-level matrix** from 5M cleaned transactions, then trains a LightGBM model to rank features by predictive power.

| Section | Description |
|---------|-------------|
| **1. Feature Engineering** | 10 feature groups aggregated per customer across 100K customers |
| **2. Feature Importance** | LightGBM regression + SHAP analysis to identify the most predictive features |

> **Input:** `output/cleaned.csv` (produced by `analysis.ipynb`)  
> **Outputs:** `output/customer_features.csv`, plots `11–13`, and `output/feature_importance_table.csv`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import lightgbm as lgb
import shap
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import LabelEncoder

%matplotlib inline

# --- Paths ---
CLEANED  = "output/cleaned.csv"
FEATURES = "output/customer_features.csv"
OUT      = "output/"

print("Setup complete.")

---
## 1. Feature Engineering

Starting from the cleaned transaction dataset (5M rows, 100K customers), we aggregate transaction-level data into **70 customer-level features** across 10 categories. Only successful transactions (4.6M) are used for behavioural features; status counts use all transactions.

| Category | # Features | Description |
|---|---|---|
| RFM | 8 | Recency (days since last txn), frequency, monetary stats (total, mean, median, std, max, min) |
| Transaction type | 4 | Debit/credit counts and ratios |
| Channel usage | 16 | Per-channel counts (7) and ratios (7), primary channel, number of channels used |
| Temporal behaviour | 8 | Active days/months, tenure, weekend ratio, mean/std hour, peak hour, day-of-week entropy |
| Transaction status | 9 | Per-status counts (4) and rates (4), total transaction count |
| Balance | 5 | Avg/max/min/std balance before transaction, balance inconsistency count |
| Geographic | 3 | Primary state, unique states visited, unique LGAs visited |
| Merchant | 2 | Unique merchants and merchant category codes |
| Monthly trends | 8 | Recent vs prior 3-month counts/amounts, trend ratios, monthly mean/std/CV for count and amount |
| Outlier | 2 | High-value transaction count and ratio |

**Output:** `customer_features.csv` — 100,000 rows × 71 columns (70 features + `customer_id`)

In [ ]:
df = pd.read_csv(CLEANED, parse_dates=["timestamp"])
print(f"Loaded: {df.shape}")

df_success = df[df["status"] == "success"].copy()
print(f"Successful transactions: {len(df_success):,}")

### Feature Design Rationale

- **RFM features** are the foundation of customer value analysis in banking. Recency captures engagement freshness, frequency captures habit strength, and monetary captures economic value.
- **Channel features** capture digital adoption and behavioural preferences — a customer who only uses mobile behaves very differently from one who visits branches.
- **Temporal features** like day-of-week entropy measure behavioural regularity. A customer with high entropy transacts evenly across the week (likely a business); low entropy suggests concentrated patterns (salary-driven spending).
- **Monthly trends** compare the most recent 3 months to the prior 3 months, capturing acceleration or decline in activity — critical for churn detection and growth identification.
- **Status features** (failed/reversed/pending rates) capture transaction reliability, which may signal account issues, insufficient funds, or fraud risk.

### 1. RFM Features

**Recency** — days since last transaction. **Frequency** — total successful transactions. **Monetary** — summary stats of transaction amount (total, mean, median, std, max, min).

In [ ]:
ref_date = df["timestamp"].max()

rfm = df_success.groupby("customer_id").agg(
    recency_days    =("timestamp", lambda x: (ref_date - x.max()).days),
    frequency       =("transaction_id", "count"),
    monetary_total  =("amount", "sum"),
    monetary_mean   =("amount", "mean"),
    monetary_median =("amount", "median"),
    monetary_std    =("amount", "std"),
    monetary_max    =("amount", "max"),
    monetary_min    =("amount", "min"),
).reset_index()
rfm["monetary_std"] = rfm["monetary_std"].fillna(0)

print(f"RFM shape: {rfm.shape}")
rfm.head(3)

### 2. Transaction Type Features

Per-customer counts and ratios of debit vs credit transactions.

In [ ]:
type_counts = df_success.groupby(["customer_id", "transaction_type"]).size().unstack(fill_value=0)
type_counts.columns = [f"txn_type_{c}_count" for c in type_counts.columns]
type_total  = type_counts.sum(axis=1)
type_ratios = type_counts.div(type_total, axis=0)
type_ratios.columns = [c.replace("_count", "_ratio") for c in type_ratios.columns]
type_feats = pd.concat([type_counts, type_ratios], axis=1).reset_index()

print(f"Type features shape: {type_feats.shape}")
type_feats.head(3)

### 3. Channel Usage Features

Per-channel counts and ratios, primary channel (most-used), and number of distinct channels used.

In [ ]:
ch_counts  = df_success.groupby(["customer_id", "channel"]).size().unstack(fill_value=0)
ch_counts.columns = [f"channel_{c}_count" for c in ch_counts.columns]
ch_total   = ch_counts.sum(axis=1)
ch_ratios  = ch_counts.div(ch_total, axis=0)
ch_ratios.columns = [c.replace("_count", "_ratio") for c in ch_ratios.columns]
ch_primary = ch_counts.idxmax(axis=1).str.replace("channel_", "").str.replace("_count", "")

ch_feats = pd.concat([ch_counts, ch_ratios], axis=1).reset_index()
ch_feats["primary_channel"] = ch_primary.values
ch_feats["n_channels_used"] = (ch_counts > 0).sum(axis=1).values

print(f"Channel features shape: {ch_feats.shape}")
ch_feats.head(3)

### 4. Temporal Behaviour Features

Covers tenure, active days/months, peak hour, weekend ratio, and day-of-week entropy (higher entropy = activity spread evenly across all days).

In [ ]:
temp = df_success.groupby("customer_id").agg(
    first_txn_date   =("timestamp", "min"),
    last_txn_date    =("timestamp", "max"),
    weekend_txn_count=("is_weekend", "sum"),
    total_txn_count  =("transaction_id", "count"),
    mean_hour        =("hour", "mean"),
    std_hour         =("hour", "std"),
    n_active_days    =("date", "nunique"),
    n_active_months  =("month", lambda x: x.nunique()),
).reset_index()

temp["tenure_days"]        = (temp["last_txn_date"] - temp["first_txn_date"]).dt.days
temp["weekend_ratio"]      = temp["weekend_txn_count"] / temp["total_txn_count"]
temp["txn_per_active_day"] = temp["total_txn_count"] / temp["n_active_days"].clip(lower=1)
temp["std_hour"]           = temp["std_hour"].fillna(0)
temp = temp.drop(columns=["first_txn_date", "last_txn_date", "weekend_txn_count", "total_txn_count"])

# Peak hour (mode)
peak_hour = (
    df_success.groupby("customer_id")["hour"]
    .agg(lambda x: x.mode().iloc[0])
    .rename("peak_hour")
)
temp = temp.merge(peak_hour, on="customer_id")

# Day-of-week entropy
dow_counts  = df_success.groupby(["customer_id", "day_of_week"]).size().unstack(fill_value=0)
dow_probs   = dow_counts.div(dow_counts.sum(axis=1), axis=0)
dow_entropy = -(dow_probs * np.log2(dow_probs.clip(lower=1e-10))).sum(axis=1)
temp["dow_entropy"] = dow_entropy.values

print(f"Temporal features shape: {temp.shape}")
temp.head(3)

### 5. Status-Based Features

Counts and rates for each transaction status (success, failed, pending, reversed) using **all** transactions (not just successful ones).

In [ ]:
status_counts   = df.groupby(["customer_id", "status"]).size().unstack(fill_value=0)
total_per_cust  = status_counts.sum(axis=1)
status_feats    = pd.DataFrame({"customer_id": status_counts.index})
status_feats["total_txn_all_status"] = total_per_cust.values
for col in status_counts.columns:
    status_feats[f"status_{col}_count"] = status_counts[col].values
    status_feats[f"status_{col}_rate"]  = (status_counts[col] / total_per_cust).values

print(f"Status features shape: {status_feats.shape}")
status_feats.head(3)

### 6. Balance Features

Summary statistics of `balance_before_ngn` and a count of balance-inconsistent transactions per customer.

In [ ]:
bal = df_success.groupby("customer_id").agg(
    avg_balance_before         =("balance_before_ngn", "mean"),
    max_balance_before         =("balance_before_ngn", "max"),
    min_balance_before         =("balance_before_ngn", "min"),
    std_balance_before         =("balance_before_ngn", "std"),
    balance_inconsistent_count =("balance_consistent", lambda x: (~x).sum()),
).reset_index()
bal["std_balance_before"] = bal["std_balance_before"].fillna(0)

print(f"Balance features shape: {bal.shape}")
bal.head(3)

### 7. Geographic Features

Primary state (mode), number of unique states, and number of unique LGAs visited.

In [ ]:
geo = df_success.groupby("customer_id").agg(
    primary_state  =("location_state", lambda x: x.mode().iloc[0]),
    n_unique_states=("location_state", "nunique"),
    n_unique_lgas  =("location_lga",   "nunique"),
).reset_index()

print(f"Geographic features shape: {geo.shape}")
geo.head(3)

### 8. Merchant Features

Number of unique merchants and unique merchant category codes (MCC) per customer. Rows where `merchant_name == 'unknown'` are excluded (P2P/internal transfers).

In [ ]:
df_merch = df_success[df_success["merchant_name"] != "unknown"]
merch = df_merch.groupby("customer_id").agg(
    n_unique_merchants=("merchant_name",          "nunique"),
    n_unique_mcc      =("merchant_category_code", "nunique"),
).reset_index()

print(f"Merchant features shape: {merch.shape}")
merch.head(3)

### 9. Monthly Trend Features

Compares the **last 3 months** vs the **prior 3 months** of activity to measure growth/decline in transaction count and amount. Also computes monthly coefficient of variation (CV) as a measure of behavioural volatility.

In [ ]:
df_success["ym"] = df_success["timestamp"].dt.to_period("M")
monthly_cust = df_success.groupby(["customer_id", "ym"]).agg(
    monthly_count =("transaction_id", "count"),
    monthly_amount=("amount",         "sum"),
).reset_index()

all_periods_sorted = sorted(monthly_cust["ym"].unique())
last_3  = all_periods_sorted[-3:]
prior_3 = all_periods_sorted[-6:-3]

recent = (
    monthly_cust[monthly_cust["ym"].isin(last_3)]
    .groupby("customer_id")
    .agg(recent_3m_count=("monthly_count", "sum"), recent_3m_amount=("monthly_amount", "sum"))
    .reset_index()
)
prior = (
    monthly_cust[monthly_cust["ym"].isin(prior_3)]
    .groupby("customer_id")
    .agg(prior_3m_count=("monthly_count", "sum"), prior_3m_amount=("monthly_amount", "sum"))
    .reset_index()
)

trend = recent.merge(prior, on="customer_id", how="outer").fillna(0)
trend["count_trend"]  = (trend["recent_3m_count"]  - trend["prior_3m_count"])  / trend["prior_3m_count"].clip(lower=1)
trend["amount_trend"] = (trend["recent_3m_amount"] - trend["prior_3m_amount"]) / trend["prior_3m_amount"].clip(lower=1)

# Monthly volatility (coefficient of variation)
monthly_vol = monthly_cust.groupby("customer_id").agg(
    monthly_count_std   =("monthly_count",  "std"),
    monthly_amount_std  =("monthly_amount", "std"),
    monthly_count_mean  =("monthly_count",  "mean"),
    monthly_amount_mean =("monthly_amount", "mean"),
).reset_index()
monthly_vol["monthly_count_cv"]  = monthly_vol["monthly_count_std"]  / monthly_vol["monthly_count_mean"].clip(lower=1)
monthly_vol["monthly_amount_cv"] = monthly_vol["monthly_amount_std"] / monthly_vol["monthly_amount_mean"].clip(lower=1)
monthly_vol = monthly_vol.fillna(0)

print(f"Trend features shape:      {trend.shape}")
print(f"Volatility features shape: {monthly_vol.shape}")
trend.head(3)

### 10. Amount Outlier Features

Count and ratio of flagged large-amount transactions (3× IQR outliers from the cleaning step).

In [ ]:
outlier_feats = df_success.groupby("customer_id").agg(
    outlier_txn_count=("amount_outlier", "sum"),
    outlier_txn_ratio=("amount_outlier", "mean"),
).reset_index()

print(f"Outlier features shape: {outlier_feats.shape}")
outlier_feats.head(3)

### Merge All Feature Groups & Save

In [ ]:
features = rfm
for feat_df in [type_feats, ch_feats, temp, status_feats, bal, geo, merch, trend, monthly_vol, outlier_feats]:
    features = features.merge(feat_df, on="customer_id", how="left")

features = features.fillna(0)

print(f"Final feature matrix: {features.shape}")
print(f"Customers:  {features['customer_id'].nunique():,}")
print(f"Features:   {features.shape[1] - 1}  (excluding customer_id)")

features.to_csv(FEATURES, index=False)
print(f"\nSaved → {FEATURES}")

---
## 2. Feature Importance

To evaluate which features carry the most signal, we train a **LightGBM regressor** to predict each customer's transaction count over the most recent 3 months (`recent_3m_count`). This target was chosen because it directly measures near-term customer activity — the quantity the bank most wants to understand and predict.

| Parameter | Value |
|---|---|
| Model | LightGBM (gradient boosted trees) |
| Target | `recent_3m_count` — 3-month transaction count |
| Features used | 64 (excluded target, its derivations, and `customer_id`) |
| Train / Test split | 80,000 / 20,000 (80/20) |
| Best iteration | 95 (early stopped from 500) |
| **Test MAE** | **1.32** |
| **Test R²** | **0.789** |

Importance is measured using two complementary methods:
- **LightGBM gain** — total reduction in loss contributed by each feature across all tree splits. Highlights features that create large improvements when used.
- **SHAP (SHapley Additive exPlanations)** — average marginal contribution of each feature to individual predictions. More stable and interpretable than gain, less biased toward high-cardinality features.

In [ ]:
df = pd.read_csv(FEATURES)
print(f"Loaded customer features: {df.shape}")
df.head(3)

### Prepare Data

Target: `recent_3m_count`. Columns that directly leak or are derived from the target are dropped. Categorical columns (`primary_channel`, `primary_state`) are label-encoded.

In [ ]:
target_col = "recent_3m_count"

drop_cols = [
    "customer_id",
    "recent_3m_count", "recent_3m_amount",   # target & correlated
    "prior_3m_count",  "prior_3m_amount",    # used to compute trend
    "count_trend",     "amount_trend",       # derived from target
]

# Encode categoricals
cat_cols = ["primary_channel", "primary_state"]
le_dict  = {}
for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    le_dict[col] = le

y = df[target_col]
X = df.drop(columns=drop_cols, errors="ignore")
feature_names = X.columns.tolist()

print(f"Features: {len(feature_names)}")
print(f"Target — {target_col}: mean={y.mean():.1f}, std={y.std():.1f}")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {len(X_train):,}  |  Test: {len(X_test):,}")

### Train LightGBM

Regressor with early stopping on MAE. 500 max rounds, patience of 50.

In [ ]:
params = {
    "objective":         "regression",
    "metric":            "mae",
    "learning_rate":     0.05,
    "num_leaves":        63,
    "min_child_samples": 50,
    "subsample":         0.8,
    "colsample_bytree":  0.8,
    "verbose":           -1,
    "seed":              42,
    "n_jobs":            -1,
}

train_data = lgb.Dataset(X_train, label=y_train)
val_data   = lgb.Dataset(X_test,  label=y_test, reference=train_data)

model = lgb.train(
    params,
    train_data,
    num_boost_round=500,
    valid_sets=[val_data],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)],
)

y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
r2  = r2_score(y_test, y_pred)
print(f"\nTest MAE: {mae:.2f}")
print(f"Test R²:  {r2:.4f}")

> **Model quality:** The model explains ~79% of variance in customer activity with an average prediction error of just 1.3 transactions over 3 months. This strong fit confirms the engineered features carry meaningful signal about customer behaviour.

### Plot 11: Built-in Feature Importance (Gain & Split)

Left: information gain (how much each feature reduces loss). Right: split count (how often each feature is used to split).

In [ ]:
importance = pd.DataFrame({
    "feature":          feature_names,
    "importance_gain":  model.feature_importance(importance_type="gain"),
    "importance_split": model.feature_importance(importance_type="split"),
})
importance = importance.sort_values("importance_gain", ascending=False).reset_index(drop=True)

top_n = 25
top   = importance.head(top_n)

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
axes[0].barh(range(top_n), top["importance_gain"].values,  color="steelblue")
axes[0].set_yticks(range(top_n))
axes[0].set_yticklabels(top["feature"].values)
axes[0].invert_yaxis()
axes[0].set_title(f"Top {top_n} Features by Gain")
axes[0].set_xlabel("Gain")

axes[1].barh(range(top_n), top["importance_split"].values, color="coral")
axes[1].set_yticks(range(top_n))
axes[1].set_yticklabels(top["feature"].values)
axes[1].invert_yaxis()
axes[1].set_title(f"Top {top_n} Features by Split Count")
axes[1].set_xlabel("Split Count")

plt.tight_layout()
plt.savefig(f"{OUT}11_feature_importance_lgbm.png", dpi=150)
plt.show()

importance.head(20)[["feature", "importance_gain", "importance_split"]]

#### Top 20 Features (SHAP rank)

| Rank (SHAP) | Feature | Mean \|SHAP\| | Gain Rank | Category |
|---|---|---|---|---|
| 1 | `frequency` | 2.072 | 8 | RFM |
| 2 | `recency_days` | 0.726 | 20 | RFM |
| 3 | `status_success_count` | 0.424 | 10 | Status |
| 4 | `total_txn_all_status` | 0.413 | 12 | Status |
| 5 | `txn_type_debit_count` | 0.343 | 4 | Txn type |
| 6 | `balance_inconsistent_count` | 0.333 | 9 | Balance |
| 7 | `status_reversed_count` | 0.331 | 1 | Status |
| 8 | `n_active_days` | 0.300 | 11 | Temporal |
| 9 | `status_pending_count` | 0.289 | 2 | Status |
| 10 | `monthly_count_mean` | 0.271 | 21 | Trends |
| 11 | `channel_agent_count` | 0.267 | 3 | Channel |
| 12 | `channel_mobile_count` | 0.163 | 15 | Channel |
| 13 | `channel_web_count` | 0.130 | 6 | Channel |
| 14 | `channel_atm_count` | 0.125 | 13 | Channel |
| 15 | `status_failed_count` | 0.077 | 7 | Status |
| 16 | `outlier_txn_count` | 0.073 | 5 | Outlier |
| 17 | `channel_pos_count` | 0.067 | 14 | Channel |
| 18 | `txn_type_credit_count` | 0.036 | 18 | Txn type |
| 19 | `tenure_days` | 0.036 | 30 | Temporal |
| 20 | `txn_per_active_day` | 0.035 | 17 | Temporal |

### Plot 12: SHAP Feature Importance (Bar)

Mean absolute SHAP values — model-agnostic measure of each feature's average impact on predictions.

In [ ]:
explainer   = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

fig, ax = plt.subplots(figsize=(10, 10))
shap.summary_plot(shap_values, X_test, plot_type="bar", max_display=25, show=False)
plt.title("SHAP Feature Importance (mean |SHAP|)")
plt.tight_layout()
plt.savefig(f"{OUT}12_shap_importance_bar.png", dpi=150, bbox_inches="tight")
plt.show()

### Plot 13: SHAP Beeswarm

Shows both the magnitude and **direction** of each feature's effect on the prediction. Each dot is one customer in the test set; colour indicates feature value (red = high, blue = low).

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
shap.summary_plot(shap_values, X_test, max_display=25, show=False)
plt.title("SHAP Beeswarm Plot")
plt.tight_layout()
plt.savefig(f"{OUT}13_shap_beeswarm.png", dpi=150, bbox_inches="tight")
plt.show()

### Feature Importance Comparison Table

Side-by-side ranking of LightGBM gain vs mean |SHAP|, sorted by SHAP rank. Saved to `output/feature_importance_table.csv`.

In [ ]:
shap_abs_mean = pd.DataFrame({
    "feature":       feature_names,
    "mean_abs_shap": np.abs(shap_values).mean(axis=0),
}).sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)

comparison = importance.merge(shap_abs_mean, on="feature")
comparison["rank_gain"] = comparison["importance_gain"].rank(ascending=False).astype(int)
comparison["rank_shap"] = comparison["mean_abs_shap"].rank(ascending=False).astype(int)
comparison = comparison.sort_values("rank_shap").reset_index(drop=True)

comparison.to_csv(f"{OUT}feature_importance_table.csv", index=False)
print(f"Saved → {OUT}feature_importance_table.csv")

comparison[["feature", "rank_shap", "mean_abs_shap", "rank_gain", "importance_gain"]].head(20)

#### Features with Zero Importance

Three features contributed nothing: `n_unique_states`, `n_unique_lgas`, and `primary_channel` (label-encoded). Geographic granularity at the state/LGA level does not help predict activity levels once other behavioural features are present. The encoded `primary_channel` was redundant given the per-channel count features.

---
## 3. Interpretation & Implications

### 3.1 Past behaviour is the strongest predictor of future behaviour

`frequency` dominates all other features by a wide margin — its SHAP value (2.07) is nearly 3× the second-place feature. This confirms the intuitive principle that **the best predictor of whether a customer will be active next quarter is how active they were historically**. For the bank, this means customer engagement programmes should prioritise maintaining existing habits rather than trying to create new ones.

`recency_days` (0.73 SHAP) reinforces this: customers who transacted recently are far more likely to continue. A growing recency gap is an early warning of disengagement.

---

### 3.2 Transaction reliability is a major behavioural signal

Four of the top 10 features are transaction status counts: `status_success_count`, `status_reversed_count`, `status_pending_count`, and `balance_inconsistent_count`. This is notable — **the quality of a customer's transaction experience, not just the quantity, strongly predicts future activity**.

- **Reversed transactions** (rank 1 by gain, rank 7 by SHAP): Customers with frequent reversals likely have unstable financial behaviour or disputed transactions. This could indicate account issues that, if unresolved, lead to churn.
- **Pending transactions** (rank 2 by gain, rank 9 by SHAP): High pending counts may signal processing delays or system issues affecting specific customer segments. If concentrated in certain channels or regions, it points to infrastructure problems.
- **Balance inconsistencies** (rank 9 by gain, rank 6 by SHAP): These were identified in EDA as exclusively tied to debit transactions. Their high importance suggests they capture meaningful behavioural variation — possibly distinguishing customers who frequently attempt transactions beyond their balance.

**Implication:** Reducing failed and pending transactions — especially on specific channels — could directly improve customer retention. The 7% failure rate from EDA is not just a service quality issue; it is a predictive signal of reduced future activity.

---

### 3.3 Channel preferences strongly differentiate customer segments

Five channel features appear in the top 20. The relative ordering is telling:

| Channel | SHAP | Interpretation |
|---|---|---|
| Agent | 0.267 | Agent banking users are a distinct segment — likely rural or underbanked customers with different activity patterns |
| Mobile | 0.163 | Mobile-heavy customers tend to be more active overall — digital adoption correlates with engagement |
| Web | 0.130 | Web users may represent business or power users making larger, less frequent transactions |
| ATM | 0.125 | ATM reliance may indicate customers less integrated into digital banking |
| POS | 0.067 | POS usage is common across segments, so it differentiates less |

**Implication:** Channel usage is not just a preference — it is a proxy for customer type, financial literacy, and geographic context. **Agent banking users** stand out as the most distinctive segment, suggesting the bank serves a meaningful population through agent networks that behaves very differently from digital-native customers. Tailored strategies for agent-dependent customers could improve retention in underserved areas.

---

### 3.4 Spending behaviour matters, but less than engagement patterns

While debit transaction count ranks 5th, the monetary features (total spend, mean amount, etc.) rank much lower — mostly outside the top 20. This reveals an important insight: **how often a customer transacts matters more than how much they spend** when predicting future activity.

This has segmentation implications: a customer making 10 small daily transactions is likely more engaged and retainable than one making 2 large monthly transfers, even if the latter has higher monetary value. Activity frequency is a better health metric than transaction value.

---

### 3.5 Temporal consistency captures engagement depth

`n_active_days` (rank 8) and `monthly_count_mean` (rank 10) both measure how spread out a customer's activity is over time. A customer with 50 transactions across 40 different days is more consistently engaged than one with 50 transactions in a 3-day burst.

`tenure_days` (rank 19) matters but less so — a long-tenured but recently inactive customer is still at risk. Tenure alone is a weak signal without recent activity.

**Implication:** Engagement regularity, not just volume, should be a key metric in customer health dashboards.

---

### 3.6 High-value transactions are a niche but informative signal

`outlier_txn_count` (rank 16 by SHAP, rank 5 by gain) shows an interesting split — it creates large gain when used in splits (suggesting it sharply separates specific customer groups) but affects fewer customers overall (lower SHAP). This likely captures **business accounts or high-net-worth individuals** who make occasional large transfers. These customers behave differently from the retail majority and may warrant separate modelling.

---

### 3.7 Geographic features are not predictive

`n_unique_states`, `n_unique_lgas`, and `primary_state` all ranked at or near the bottom. This does not mean geography is unimportant for the business — it means that **once you know a customer's behavioural features (channels, frequency, amounts), knowing their location adds little predictive power**. Geographic effects are already captured indirectly through channel preferences (agent banking correlates with rural areas) and merchant patterns.

---
## 4. Gain vs. SHAP: Why the Rankings Differ

The two methods sometimes rank features very differently. Key discrepancies:

| Feature | Gain Rank | SHAP Rank | Explanation |
|---|---|---|---|
| `status_reversed_count` | 1 | 7 | High gain because it creates sharp splits for a small subset of customers with reversals. Lower SHAP because most customers have zero reversals, so it affects few predictions. |
| `recency_days` | 20 | 2 | Low gain because it makes many small contributions across hundreds of splits (689 — the most of any feature). High SHAP because those small contributions add up to a large cumulative effect. |
| `frequency` | 8 | 1 | Similar to recency — used in many splits (388) with moderate individual gain, but the cumulative SHAP effect is dominant. |
| `channel_agent_count` | 3 | 11 | High gain for the small agent-user segment; lower average SHAP because most customers don't use agent banking. |

**Takeaway:** Gain highlights features that are powerful for specific subgroups. SHAP highlights features that matter broadly across the population. Both perspectives are valuable — gain identifies niche differentiators while SHAP identifies universally important drivers.

---
## 5. Key Takeaways for Downstream Modelling

1. **For time series prediction:** The strong R² (0.789) using customer-level features validates that behavioural aggregates can predict near-term activity. For transaction-level time series, lag features and rolling windows should be built around the top features here — particularly frequency, recency, and monthly cadence.

2. **For customer segmentation:** The feature importance ranking provides a natural feature selection guide. The top ~20 features capture the vast majority of signal. Segmentation should emphasise:
   - Activity level (`frequency`, `n_active_days`, `monthly_count_mean`)
   - Channel profile (mobile vs. agent vs. ATM dominance)
   - Transaction reliability (failure/reversal rates)
   - Spending intensity (`txn_type_debit_count`, `outlier_txn_ratio`)

3. **For churn modelling:** Recency, `count_trend` (recent vs. prior 3 months), and transaction failure rates are the most actionable churn indicators. A customer whose recency is growing, trend is declining, and failure rate is rising is at high risk.

4. **Feature reduction:** The bottom 30 features collectively contribute less SHAP importance than `frequency` alone. For production models, the top 20–25 features are likely sufficient, reducing complexity without sacrificing accuracy.

---
## Model Summary

In [ ]:
summary = {
    "Model":              "LightGBM Regressor",
    "Target":             f"{target_col} (3-month transaction count)",
    "Features used":      len(feature_names),
    "Train size":         f"{len(X_train):,}",
    "Test size":          f"{len(X_test):,}",
    "Best iteration":     model.best_iteration,
    "Test MAE":           f"{mae:.2f}",
    "Test R²":            f"{r2:.4f}",
    "Top feature (SHAP)": shap_abs_mean.iloc[0]["feature"],
    "Top feature (gain)": importance.iloc[0]["feature"],
}

pd.DataFrame.from_dict(summary, orient="index", columns=["Value"])

### Plots Reference

| File | Description |
|---|---|
| `11_feature_importance_lgbm.png` | Top 25 features by LightGBM gain and split count |
| `12_shap_importance_bar.png` | Top 25 features by mean absolute SHAP value |
| `13_shap_beeswarm.png` | SHAP beeswarm plot — shows direction and magnitude of feature effects |
| `feature_importance_table.csv` | Full ranking table with gain, split count, SHAP, and ranks |